#대학 등록금 데이터 전처리 템플릿 (2010~2023)

In [ ]:
import pandas as pd
from pathlib import Path
# 파일 경로를 수동으로 지정
file_paths = [
    "2010년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2011년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2012년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2013년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2014년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2015년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2016년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2017년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2018년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2019년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2020년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2021년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2022년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx",
    "2023년 _대학_8-차-1. 등록금 현황_학교별자료.xlsx"
]
file_paths

In [ ]:

# 사이버대 제거 키워드
cyber_keywords = ['사이버', '방송통신']
dfs = {}

for path in file_paths:
    try:
        year = int(Path(path).stem[:4])
        df = pd.read_excel(path, header=3)
        df = df[~df['학교'].str.contains('|'.join(cyber_keywords), na=False)]
        df = df.rename(columns={'학교': '학교명'})
        df['기준년도'] = year
        dfs[year] = df
    except Exception as e:
        print(f"X {path} 처리 중 오류 발생:", e)

In [ ]:

df_2023 = dfs[2023]
school_to_region = df_2023.set_index('학교명')['지역'].to_dict()
school_to_type = df_2023.set_index('학교명')['설립구분'].to_dict()

processed = []
school_sets = [set(df['학교명']) for df in dfs.values()]
common_schools = set.intersection(*school_sets) if school_sets else set()
계열_컬럼 = ['수업료\n(B)', '인문사회', '자연과학', '예체능', '공학', '의학']


for year, df in dfs.items():
    df = df[df['학교명'].isin(common_schools)].copy()

    # 등록금 선택
    if '등록금\n(D=B+C)' in df.columns:
        df['등록금'] = df['등록금\n(D=B+C)']
    elif '등록금\n(D=B)' in df.columns:
        df['등록금'] = df['등록금\n(D=B)']
    else:
        df['등록금'] = 0

    # 입학금 보완 후 소수점 반올림, 단위 보정은 연도 기준 적용
    if '입학금\n(A)' in df.columns:
        df['입학금\n(A)'] = pd.to_numeric(df['입학금\n(A)'], errors='coerce').fillna(0)
    else:
        df['입학금\n(A)'] = pd.Series(0, index=df.index)

    df['등록금'] = pd.to_numeric(df['등록금'], errors='coerce').fillna(0) + df['입학금\n(A)']

    #  연도별 단위 확인: 2010~2015년은 원 단위, 이후는 천원 단위라고 가정하여 보정
    if year <= 2015:
        df['등록금'] = df['등록금'].round(1)  # 2016년부터는 천원 단위이므로 반올림만
    else:
        df['등록금'] = (df['등록금'] / 1000).round(1)  # 그 전은 원 단위 → 천원 환산
    # 수업료/계열별 단위 정리
  
    for col in 계열_컬럼:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
            if year >= 2020:
                df[col] = (df[col] / 1000).round(1)  # 2020~2023은 원 → 천원 환산
            else:
                df[col] = df[col].round(1)  # 그 외는 천원 단위 유지

    # 학교명 기준 지역/설립구분 매핑
    if '지역' not in df.columns:
        df['지역'] = df['학교명'].map(school_to_region)
    else:
        df['지역'] = df['학교명'].map(school_to_region).combine_first(df['지역'])

    if '설립구분' not in df.columns:
        df['설립구분'] = df['학교명'].map(school_to_type)
    else:
        df['설립구분'] = df['학교명'].map(school_to_type).combine_first(df['설립구분'])

    keep_cols = ['학교명', '기준년도', '등록금', '지역', '설립구분',
                 '수업료\n(B)', '인문사회', '자연과학', '예체능', '공학', '의학']
    for col in keep_cols:
        if col not in df.columns:
            df[col] = pd.NA

    df = df[keep_cols]
    processed.append(df)

final_df = pd.concat(processed, ignore_index=True)
final_df.to_csv("최종_등록금_통합_2010_2023.csv", index=False, encoding="utf-8-sig")
final_df.head()
